# 01 · NetFlow v3 data stage (NF-UNSW-NB15-v3, NF-ToN-IoT-v3, NF-CSE-CIC-IDS2018-v3)

Design decisions:
- Source: NetFlow v3 datasets from the University of Queensland (Luay et al. 2025, arXiv 2503.04404), 53 NetFlow features.
- Datasets: NF-UNSW-NB15-v3 and NF-ToN-IoT-v3 (first certificate experiment) and NF-CSE-CIC-IDS2018-v3 (confirmation stage; added 30 Sep 2026, before any certificate result, because it comes from a different research group and network than the other two). NF-BoT-IoT-v3 is excluded: 0.3% of its flows are benign, too few to test certificates on benign verdicts.
- Raw archives stay in the Drive project folder, outside the repository; processed parquet files go to `XIDS_Certify_Research/data/nf3_v1/`.
- Model inputs: 46 features. Excluded as provenance: IPv4 source and destination addresses, source and destination ports, flow start and end timestamps. `DNS_QUERY_ID` is also excluded, because it is a random transaction identifier.
- Label: the binary `Label` column. Rows whose `Label` contradicts the `Attack` column are quarantined.
- Zero-duration flows: nProbe leaves the two per-second byte rates empty when a flow lasts 0 ms. The rate is undefined, so it is set to 0 and the flow is kept; the zero duration itself stays visible to the model. Any other row with a missing or non-finite model feature is dropped and counted.
- Exact model-feature duplicates collapse to one row with a multiplicity count. A duplicate group whose copies disagree on the label keeps its majority label when at least 99% of its copies agree (threshold fixed before any model is trained) and is quarantined otherwise. Label purity and the minority count are stored for every row.
- Clusters: conversation-minute groups (source address, destination address, destination port, protocol, 60 s window of the flow start). A cluster never crosses roles.
- Four disjoint roles: detector_train 0.50, development 0.15, calibration 0.15, locked_test 0.20, stratified by attack family. Each dataset is split on its own, so adding a dataset never changes another dataset's split.
- A dataset whose parquet file already exists is not reprocessed, and its committed tables are kept, unless the data-stage configuration has changed since that run (then every dataset is reprocessed) or `FORCE = True`.
- Each CSV is read twice in 1M-row chunks, so memory stays bounded for the large files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, importlib, json, os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
assert (REPO / 'tools' / 'commit_cell.py').exists(), 'tools/commit_cell.py is missing from the repo'

import numpy as np
import pandas as pd
import src.nf3_data as nf
importlib.reload(nf)

DATASETS = ['unsw', 'ton', 'cic2018']
FORCE = False                                # True reprocesses every dataset from its archive
WORK = Path('/content/nf3_work')             # runtime disk: fast reads for the large archives
TABLES = REPO / 'results' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
OUT = PROJECT / 'data' / 'nf3_v1'
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

config_hash = nf.write_config(REPO)
previous = json.loads((TABLES / 'nf01_summary.json').read_text()) if (TABLES / 'nf01_summary.json').exists() else {}
if previous and previous.get('config_sha256') != config_hash:
    print('data-stage configuration changed since the last run: every dataset will be reprocessed')
    FORCE = True
TODO = [d for d in DATASETS if FORCE or not (OUT / d / 'flows.parquet').exists() or d not in previous.get('split_hash', {})]
print(f'pandas {pd.__version__} | numpy {np.__version__} | model features: {len(nf.MODEL_FEATURES)}')
print(f'to process: {TODO} | kept from the committed run: {[d for d in DATASETS if d not in TODO]}')

## Archives: identify each file from the CSV inside it

In [ ]:
archives = nf.find_archives(PROJECT, REPO)
display(archives[['file', 'member', 'dataset', 'bytes', 'member_bytes']])
found = archives.dropna(subset=['dataset'])
missing = set(DATASETS) - set(found['dataset'])
assert not missing, f'no archive found for: {sorted(missing)} (is the upload finished?)'
assert not found[found['dataset'].isin(DATASETS)]['dataset'].duplicated().any(), 'two archives hold the same dataset'
SRC = {r.dataset: (Path(r.path), r.member) for r in found.itertuples() if r.dataset in DATASETS}

## One dataset at a time: schema, labels, duplicates, clusters, roles, parquet

In [ ]:
RESULTS = {}

def run(ds):
    if ds not in TODO:
        print(f'{ds}: parquet and tables from the committed run are kept (split hash {previous["split_hash"][ds][:16]}...)')
        return
    local = nf.stage_local(SRC[ds][0], WORK)
    manifest = {'dataset': ds, 'file': SRC[ds][0].name, 'member': SRC[ds][1],
                'bytes': local.stat().st_size, 'sha256': nf.sha256_file(local)}
    arrays, fams, schema, info = nf.scan(local, SRC[ds][1], ds)
    plan, audit, by_role, by_fam, labels = nf.decide(arrays, fams, ds)
    del arrays
    gc.collect()
    nf.write_parquet(local, SRC[ds][1], plan, fams, ds, OUT / ds)
    check = nf.load_split(PROJECT, ds, columns=['cluster_id', 'role'])
    assert len(check) == audit['rows_after_collapse']
    assert check.groupby('cluster_id')['role'].nunique().max() == 1
    print(f"{ds}: {len(check):,} rows written to {OUT / ds / 'flows.parquet'}")
    RESULTS[ds] = {'manifest': manifest, 'schema': schema, 'info': info, 'audit': audit,
                   'by_role': by_role, 'by_family': by_fam, 'labels': labels}
    print(info)
    display(labels)
    display(pd.Series(audit).to_frame('value'))
    display(by_role)
    display(by_fam)
    display(schema[schema['constant'] | (schema['n_missing'] > 0)])
    del plan, check
    local.unlink()
    gc.collect()

In [ ]:
run('unsw')

In [ ]:
run('ton')

In [ ]:
run('cic2018')

## Tables, split hashes and configuration (rows for kept datasets are carried over unchanged)

In [ ]:
def merge(name, key, new_frames):
    path = TABLES / name
    old = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(old) and key in old.columns:
        old = old[~old[key].isin(list(RESULTS))]
    out = pd.concat([old] + new_frames, ignore_index=True) if new_frames else old
    out = out.sort_values(key, key=lambda s: s.map({d: i for i, d in enumerate(DATASETS)}), kind='stable')
    out.to_csv(path, index=False)
    return out

new = lambda part: [RESULTS[d][part] for d in DATASETS if d in RESULTS]
merge('nf01_archives.csv', 'dataset', [pd.DataFrame([RESULTS[d]['manifest']]) for d in DATASETS if d in RESULTS])
merge('nf01_schema.csv', 'dataset', new('schema'))
merge('nf01_labels.csv', 'dataset', new('labels'))
merge('nf01_split_by_role.csv', 'dataset', new('by_role'))
merge('nf01_split_by_family.csv', 'dataset', new('by_family'))
audit = merge('nf01_split_audit.csv', 'dataset', [pd.DataFrame([{**RESULTS[d]['info'], **RESULTS[d]['audit']}]) for d in DATASETS if d in RESULTS])

summary = {'config': 'configs/data_nf3_v1.json', 'config_sha256': config_hash}
for field, src in (('rows_read', 'info'), ('rows_published', 'info'), ('rows_after_collapse', 'audit'), ('split_hash', 'audit')):
    summary[field] = {d: (RESULTS[d][src][field] if d in RESULTS else previous[field][d]) for d in DATASETS}
summary['model_features'] = len(nf.MODEL_FEATURES)
(TABLES / 'nf01_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
display(audit.set_index('dataset').T)
print(json.dumps(summary, indent=2))

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 01: NF-CSE-CIC-IDS2018-v3 added as the third NetFlow v3 dataset (decided before any certificate result) with the same data-stage rules; NF-UNSW-NB15-v3 and NF-ToN-IoT-v3 kept unchanged (per-dataset splits, same configuration hash); tables merged across the three datasets"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)